In [3]:
import argparse
from datetime import datetime, timezone
import pandas as pd

from aind_data_schema.components.identifiers import Code, DataAsset
from aind_data_schema.core.processing import (
    DataProcess,
    Processing,
    ProcessName,
    ProcessStage,
    ResourceTimestamped,
    ResourceUsage,
)
from aind_data_schema_models.units import MemoryUnit
from aind_data_schema_models.system_architecture import OperatingSystem, CPUArchitecture
from codeocean import CodeOcean
import os
# Resolve code/beh_ephys_analysis (the folder containing `utils`) relative to this
# file's location, so imports work no matter where the repo is checked out.
import os
import sys
_beh_ephys_root = os.path.normpath(os.path.join(os.getcwd(), 'beh_ephys_analysis'))
if _beh_ephys_root not in sys.path:
    sys.path.insert(0, _beh_ephys_root)
from utils.beh_functions import parseSessionID
import json
from utils.capsule_migration import CAPSULE_ROOT

# Generating fiber location data's procesing metadata

In [4]:
client = CodeOcean(domain="https://codeocean.allenneuraldynamics.org", token=os.getenv("API_SECRET"))

In [5]:
computation_params_file = CAPSULE_ROOT + '/code/data_management/processing_params.json'
with open(computation_params_file, 'r') as f:
    computation_params = json.load(f)

## CCF for fibers

In [6]:
ccf_data_asset_id = '66668438-3b32-4160-b571-037e117d8568'
ts = client.data_assets.get_data_asset(data_asset_id=ccf_data_asset_id).created
t = datetime.fromtimestamp(ts, tz=timezone.utc)

In [7]:
data_asset_ids = ['fa0f5657-6e45-41ce-a731-ca09e8233fcc', 
                '95af7045-866e-45b5-9cc4-b0c0bde67370',
                '2ad47cc1-9d30-489c-b167-c2dd309c2e70',
                '432f2f21-61c8-4809-9a83-bb9a0f22d867', 
                'f9b58f5d-af5a-4f64-a613-249e35b6b932',
                '934efb5b-b62d-4fc8-ab0b-8dc61e927008',
                '1d08a082-a3e9-40c5-8110-7aa61d1ceb4d',
                '9a1188e9-cd41-4fbd-a593-fa9db7457463',
                '861fa6c1-f9d9-4fc0-bb0f-8a61056e5546']

data_asset_names_full = [client.data_assets.get_data_asset(data_asset_id=id).name  for id in data_asset_ids]
data_assets = [DataAsset(url=name) for name in data_asset_names_full]

In [8]:
dp_list = []
# generate location files for each animal
curr_code = Code(
    url='https://github.com/AllenNeuralDynamics/ccf_annotations_data_prep',
    run_script='code/photometry.ipynb',
    version='da10a6e895b00f7722b834a092c14b95f434ecc7',
    input_data=data_assets,
)
curr_dp = DataProcess(
            process_type=ProcessName.OTHER,
            name='Convert manually labeling to ccf locations',
            experimenters=["Sue Su"],
            stage=ProcessStage.ANALYSIS,
            start_date_time=t,
            output_path='',
            code=curr_code,
            notes='Convert manually labeling to ccf locations',
            )
dp_list.append(curr_dp)
# generate combined csv file
curr_code = Code(
    url='https://github.com/AllenNeuralDynamics/ccf_annotations_data_prep',
    run_script='code/combine_photometry.ipynb',
    version='da10a6e895b00f7722b834a092c14b95f434ecc7',
)
curr_dp = DataProcess(
            process_type=ProcessName.OTHER,
            name="Combine different animals' fiber location",
            experimenters=["Sue Su"],
            stage=ProcessStage.ANALYSIS,
            start_date_time=t,
            output_path='PL_ccf_coordinates_pir.csv',
            code=curr_code,
            notes="Combine different animals' fiber location",
            )
dp_list.append(curr_dp) 

p_ccf_meta = Processing.create_with_sequential_process_graph(
    data_processes=dp_list)